# Company dataset profile
Read-only evidence for the Phase 1A schema proposal, 15 October 2026. Run with Python and openpyxl from the bundled workspace runtime. Never saves Excel files or accesses a database. Only aggregate counts and worksheet references are output. Source grain is a register entry, not a verified unique company. No time history is available.


In [1]:
"""Read-only schema evidence; output contains no contact names or phone digits."""
import hashlib
import json
import re
import unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import openpyxl

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "company-dataset.xlsx").is_file())
source = root / "company-dataset.xlsx"
before_hash = hashlib.sha256(source.read_bytes()).hexdigest()
book = openpyxl.load_workbook(source, read_only=True, data_only=False)
assert book.sheetnames == ["Sheet1"]
physical = list(book["Sheet1"].iter_rows())
headers = [c.value for c in physical[0]]
data = [(r, tuple(c.value for c in row)) for r, row in enumerate(physical[1:], 2)
        if any(c.value is not None for c in row)]


def norm(value):
    return " ".join(unicodedata.normalize("NFKC", str(value or "")).split()).casefold()


def duplicate_groups(key):
    groups = defaultdict(list)
    for row, values in data:
        value = key(values)
        if value:
            groups[value].append(row)
    return [rows for rows in groups.values() if len(rows) > 1]


columns = []
for c, header in enumerate(headers):
    present = [values[c] for _, values in data if norm(values[c])]
    missing = len(data) - len(present)
    columns.append({
        "column": header, "excel_column": chr(65 + c), "nonempty": len(present),
        "missing": missing, "missing_pct": round(missing / len(data) * 100, 2),
        "distinct_raw_nonempty": len(set(present)),
        "distinct_normalized_nonempty": len(set(map(norm, present))),
        "nonempty_types": dict(Counter(type(v).__name__ for v in present)),
        "outer_whitespace_cells": sum(isinstance(v, str) and v != v.strip() for v in present),
        "sentinel_cells": sum(norm(v) in {"n/a", "na", "none", "null", "unknown", "-", "nil", "not available"} for v in present),
    })
phones = [(row, chr(65 + c), str(v[c])) for row, v in data for c in (3, 4) if norm(v[c])]
phone_groups = defaultdict(list)
for row, column, value in phones:
    phone_groups[re.sub(r"\D", "", value)].append({"row": row, "column": column})
repeated_phones = [cells for value, cells in phone_groups.items() if value and len(cells) > 1]
names = {row: norm(v[1]) for row, v in data}
near_pairs = []
for pos, (ra, va) in enumerate(data):
    for rb, vb in data[pos + 1:]:
        a, b = [re.sub(r"[^\w]", "", norm(v[1])) for v in (va, vb)]
        score = SequenceMatcher(None, a, b).ratio()
        if a != b and score >= 0.82:
            near_pairs.append({"rows": [ra, rb], "similarity": round(score, 4)})
suffixes = [int(v[0].split("-")[-1]) for _, v in data]
profile = {
    "source": {"file": source.name, "sheet": "Sheet1", "range": "A1:E181", "sha256": before_hash},
    "method": {"missing": "None or whitespace-only string",
               "normalization": "Unicode NFKC, collapse whitespace, casefold",
               "near_names": "Remove non-word characters after normalization; difflib SequenceMatcher >= 0.82. Heuristic, not merge evidence.",
               "privacy": "Aggregates and 1-based Excel row/column references only; no contact names or phone digits."},
    "data_rows": len(data), "physical_rows_including_header": len(physical), "columns": columns,
    "register_ids": {"duplicates": duplicate_groups(lambda v: norm(v[0])),
                     "all_match_reg_digits": all(re.fullmatch(r"REG-\d+", v[0]) for _, v in data),
                     "minimum_suffix": min(suffixes), "maximum_suffix": max(suffixes),
                     "missing_suffixes": sorted(set(range(min(suffixes), max(suffixes) + 1)) - set(suffixes))},
    "duplicates": {"full_raw_row_groups": duplicate_groups(lambda v: v),
                   "raw_company_name_groups": duplicate_groups(lambda v: v[1]),
                   "normalized_company_name_groups": duplicate_groups(lambda v: norm(v[1])),
                   "raw_payload_excluding_register_id_groups": duplicate_groups(lambda v: v[1:]),
                   "normalized_payload_excluding_register_id_groups": duplicate_groups(lambda v: tuple(map(norm, v[1:]))),
                   "near_name_candidate_pairs": near_pairs},
    "contacts": {"missing_rows": [r for r, v in data if not norm(v[2])],
                 "delimiter_candidate_rows": [r for r, v in data if v[2] and re.search(r"[/&;]", v[2])]},
    "phones": {"nonempty_cells": len(phones), "distinct_digit_strings": len(phone_groups),
               "all_digits_only": all(re.fullmatch(r"\d+", v) is not None for _, _, v in phones),
               "length_counts": dict(Counter(len(v) for _, _, v in phones)),
               "leading_zero_length_counts": dict(Counter(len(v) for _, _, v in phones if v.startswith("0"))),
               "population": dict(Counter("both" if norm(v[3]) and norm(v[4]) else "phone1_only" if norm(v[3]) else "phone2_only" if norm(v[4]) else "neither" for _, v in data)),
               "repeated_digit_groups": repeated_phones,
               "repeated_groups_with_different_names": sum(len({names[c["row"]] for c in cells}) > 1 for cells in repeated_phones)},
    "neither_contact_nor_phone_rows": [r for r, v in data if not any(norm(v[c]) for c in (2, 3, 4))],
    "formula_cells": [c.coordinate for row in physical for c in row if c.data_type == "f"],
    "error_cells": [c.coordinate for row in physical for c in row if c.data_type == "e"],
    "address_column_present": any("address" in str(h).lower() for h in headers),
    "date_column_present": any("date" in str(h).lower() for h in headers),
}
book.close()
assert hashlib.sha256(source.read_bytes()).hexdigest() == before_hash
assert profile["data_rows"] == 180
assert [c["missing"] for c in columns] == [0, 0, 17, 69, 175]
assert profile["register_ids"]["duplicates"] == []
assert len(profile["duplicates"]["normalized_company_name_groups"]) == 14
assert len(profile["duplicates"]["raw_payload_excluding_register_id_groups"]) == 2
assert len(profile["duplicates"]["normalized_payload_excluding_register_id_groups"]) == 3
assert len(phones) == 116 and len(phone_groups) == 106
assert len(repeated_phones) == 10
assert profile["phones"]["repeated_groups_with_different_names"] == 7
assert len(profile["contacts"]["delimiter_candidate_rows"]) == 10
assert len(profile["neither_contact_nor_phone_rows"]) == 13
assert profile["formula_cells"] == profile["error_cells"] == []
print(json.dumps(profile, indent=2))


{
  "source": {
    "file": "company-dataset.xlsx",
    "sheet": "Sheet1",
    "range": "A1:E181",
    "sha256": "97936c31d1bc52ccad2f78a347c741118e1557204066ec63b56e0102aee342c0"
  },
  "method": {
    "missing": "None or whitespace-only string",
    "normalization": "Unicode NFKC, collapse whitespace, casefold",
    "near_names": "Remove non-word characters after normalization; difflib SequenceMatcher >= 0.82. Heuristic, not merge evidence.",
    "privacy": "Aggregates and 1-based Excel row/column references only; no contact names or phone digits."
  },
  "data_rows": 180,
  "physical_rows_including_header": 181,
  "columns": [
    {
      "column": "Register ID",
      "excel_column": "A",
      "nonempty": 180,
      "missing": 0,
      "missing_pct": 0.0,
      "distinct_raw_nonempty": 180,
      "distinct_normalized_nonempty": 180,
      "nonempty_types": {
        "str": 180
      },
      "outer_whitespace_cells": 0,
      "sentinel_cells": 0
    },
    {
      "column": "Compa